In [2]:
import numpy as np

In [3]:
import pandas as pd

In [4]:
import requests

In [5]:
url = "https://api.worldbank.org/v2/country/all/indicator/SI.POV.GINI?format=json&per_page=20000"

In [6]:
response = requests.get(url)

In [7]:
response.status_code

200

In [8]:
data=response.json()

In [10]:
gini = pd.DataFrame(data[1])

In [13]:
gini = gini.rename(columns={
    "country": "Country",
    "countryiso3code": "Country_Code",
    "date": "Year",
    "value": "Gini"
})

In [20]:
gini["Country"] = gini["Country"].apply(lambda x: x["value"])

In [21]:
gini["Country"].head(20)

0     Africa Eastern and Southern
1     Africa Eastern and Southern
2     Africa Eastern and Southern
3     Africa Eastern and Southern
4     Africa Eastern and Southern
5     Africa Eastern and Southern
6     Africa Eastern and Southern
7     Africa Eastern and Southern
8     Africa Eastern and Southern
9     Africa Eastern and Southern
10    Africa Eastern and Southern
11    Africa Eastern and Southern
12    Africa Eastern and Southern
13    Africa Eastern and Southern
14    Africa Eastern and Southern
15    Africa Eastern and Southern
16    Africa Eastern and Southern
17    Africa Eastern and Southern
18    Africa Eastern and Southern
19    Africa Eastern and Southern
Name: Country, dtype: str

In [30]:
gini["Year"] = pd.to_numeric(gini["Year"])

In [32]:
gini_clean = gini.dropna(subset=["Gini"]).copy()

In [37]:
country_url = "https://api.worldbank.org/v2/country?format=json&per_page=400"

In [38]:
country_response = requests.get(country_url)

In [39]:
country_data = country_response.json()

In [41]:
countries = pd.DataFrame(country_data[1])

In [43]:
countries["Region"] = countries["region"].apply(lambda x: x["value"])

In [44]:
countries[["id", "name", "Region"]].head()

,id,name,Region
0,ABW,Aruba,Latin America & Caribbean
1,AFE,Africa Eastern and Southern,Aggregates
2,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan"
3,AFR,Africa,Aggregates
4,AFW,Africa Western and Central,Aggregates


In [45]:
countries_only = countries[countries["Region"] != "Aggregates"].copy()

In [46]:
countries_only.shape

(217, 11)

In [47]:
gini_clean = gini_clean.merge(
    countries_only[["id", "Region"]],
    left_on="Country_Code",
    right_on="id",
    how="inner"
)

In [48]:
gini_clean = gini_clean.drop(columns=["id"])

In [49]:
gini_clean.head()

,indicator,Country,Country_Code,Year,Gini,unit,obs_status,decimal,Region
0,"{'id': 'SI.POV.GINI', 'value': 'Gini index'}",Albania,ALB,2020,29.4,,,1,Europe & Central Asia
1,"{'id': 'SI.POV.GINI', 'value': 'Gini index'}",Albania,ALB,2019,30.1,,,1,Europe & Central Asia
2,"{'id': 'SI.POV.GINI', 'value': 'Gini index'}",Albania,ALB,2018,30.1,,,1,Europe & Central Asia
3,"{'id': 'SI.POV.GINI', 'value': 'Gini index'}",Albania,ALB,2017,33.1,,,1,Europe & Central Asia
4,"{'id': 'SI.POV.GINI', 'value': 'Gini index'}",Albania,ALB,2016,33.7,,,1,Europe & Central Asia


In [50]:
gini_clean.shape

(2430, 9)

In [51]:
gini_clean["indicator"] = gini_clean["indicator"].apply(lambda x: x["value"])

In [53]:
gini_analysis = gini_clean[
    ["Country", "Country_Code", "Year", "Gini", "Region"]
].copy()

In [54]:
gini_analysis.head()

,Country,Country_Code,Year,Gini,Region
0,Albania,ALB,2020,29.4,Europe & Central Asia
1,Albania,ALB,2019,30.1,Europe & Central Asia
2,Albania,ALB,2018,30.1,Europe & Central Asia
3,Albania,ALB,2017,33.1,Europe & Central Asia
4,Albania,ALB,2016,33.7,Europe & Central Asia


In [57]:
gini_analysis.duplicated(
    subset=["Country_Code", "Year"]
).sum()

np.int64(0)

In [58]:
gini_analysis["Gini"].min()

np.float64(20.2)

In [59]:
gini_analysis.to_csv(
    "../data/gini_clean.csv",
    index=False
)